### This is a trial to run the meta-learning agent in the trial environment
Population weights: Uniform

In [1]:
import os, sys
os.environ["PYTHONPATH"] = "/home/rifnyrachman7"
sys.path.append("/home/rifnyrachman7/_merlion/")
sys.path.append("/home/rifnyrachman7/_merlion/ds-project-messiah/src")
sys.path.append("/home/rifnyrachman7/")

In [2]:
from gymnasium.wrappers import TimeLimit
import gymnasium as gym
from pathlib import Path

# Import both modules
from _metamorl.algo_ray.rllib.algorithms.maml.merlion_main_multitasks import MERLION, MERLIONConfig
from _metamorl.algo_ray.rllib.algorithms.maml.callback_save_population import _find_archive , MERLIONWithSaveCallbacks

 #v2.3.1 #use algo from local repo
import ray
from ray import air, tune
from ray.tune.registry import register_env
from gymnasium.wrappers import TimeLimit
from ray.air import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

# Import the required modules

from messiah.environment import Environment
from messiah.agents.processing import MaxAvailableProcessingAgent

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

import tempfile
import pickle
import json


#from utils.plot_utils import PlotUtils

In [3]:
#import generated environment
from _merlion.random_env import complex_env_merlion
#complex_sc = complex_env_merlion.ComplexSC()
#simple_sc.save_filename = "/home/rifnyrachman7/_merlion/random_env/simple_reward.pkl"

In [4]:
def make_env(env_config=None):
    base = ComplexSC()
    base = ScalarizationWeightProxy(base)
    base = TimeLimit(base, max_episode_steps=100)   # outermost!
    return base

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
def save_full_checkpoints_for_population(algo, out_dir: str):
    archive = _find_archive(algo)
    if archive is None:
        print("[SavePopulation] MERLION archive not found on algorithm; skipping.")
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    pol = algo.get_policy()
    P = int(getattr(archive, "population_size", len(getattr(archive, "archive", []))))

    for i in range(P):
        item = archive.get_archive_item(i)
        if item is None:
            continue

        theta_i = item["theta"]
        w_i = np.asarray(item["weight"], dtype=np.float32)   # <- correct key

        # save the full RLlib checkpoint for policy i
        pol.set_weights(theta_i)
        ckpt_dir = out / f"checkpoint_policy_{i}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        path = algo.save(str(ckpt_dir))
        print(f"[SavePopulation] Saved RLlib checkpoint for i={i} → {path}")

        # save the associated weight next to it
        with open(ckpt_dir / "weight.json", "w") as f:
            json.dump([float(x) for x in w_i.ravel().tolist()], f)

In [7]:
ray.shutdown()
ray.init()

register_env("complex_sc", lambda cfg: complex_env_merlion.make_env(cfg))

rollout_fragment_length = 32 #default 32

2026-02-21 22:36:46,204	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-02-21 22:36:46,206	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-02-21 22:36:46,207	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

### Adaptation and Metalearning Phase

In [8]:
seed_base = 99

config = (
    MERLIONConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
    )
    .framework("torch")
    .environment("complex_sc", clip_actions=True)
    .training(
        inner_adaptation_steps=8,
        meta_batch_size=4, # follow rollout_workers in MIRACL that represent task batch number
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
        
        #MERLION specific parameters
        population_size = 21, #default # TO DO: ablation for this
        num_objectives = 3,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
    .callbacks(MERLIONWithSaveCallbacks)
)

In [9]:
local_dir = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0"
num_iterations = 13 #default 500

results = {}
runs = [6] #run multiple times for model robustness
for k in runs:
    print(f"RUNNING NUMBER-{k}")
    
    tuner = tune.Tuner(
        MERLION,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            name=f"merlion_complex_{k}",
            local_dir = local_dir,
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),
            log_to_file=True,
            checkpoint_config=air.CheckpointConfig(checkpoint_at_end=True, num_to_keep=1),  # ensure a checkpoint exists
        )
    )
    results[k] = tuner.fit()
    
    # #save the best checkpoint for each run
    best = results[k].get_best_result()
    algo = Algorithm.from_checkpoint(best.checkpoint)
    
    out_dir = os.path.join(local_dir, f"metapolicies-{k}")
    os.makedirs(out_dir, exist_ok=True)
    save_full_checkpoints_for_population(algo, out_dir)

RUNNING NUMBER-6


/home/rifnyrachman7/.local/lib/python3.10/site-packages/ray/tune/execution/trial_runner.py:402: UserWarning: fail_fast='raise' detected. Be careful when using this mode as resources (such as Ray processes, file descriptors, and temporary files) may not be cleaned up properly. To use a safer mode, use fail_fast=True.
  warnings.warn(
(MERLION pid=1915) 2026-02-19 04:18:09,326	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=1915) 2026-02-19 04:18:09,326	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=1915) 2026-02-19 04:18:09,828	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
(MERLION pid=1915) 2026-02-19 04:18:09,828	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' 

(MERLION pid=1915) 
(MERLION pid=1915) === POPULATION OBJECTIVE BREAKDOWN (Iter 0) ===
(MERLION pid=1915) 00 w=[0.205,0.707,0.088] obj=[   0.24,   0.77,   0.41] sc=    0.631  <- OK
(MERLION pid=1915) 01 w=[0.163,0.086,0.751] obj=[   0.16,   0.79,   0.47] sc=    0.447  <- OK
(MERLION pid=1915) 02 w=[0.022,0.746,0.232] obj=[   0.19,   0.75,   0.13] sc=    0.593  <- OK
(MERLION pid=1915) 03 w=[0.453,0.092,0.456] obj=[   0.22,   0.78,   0.24] sc=    0.276  <- OK
(MERLION pid=1915) 04 w=[0.405,0.012,0.582] obj=[   0.19,   0.76,   0.22] sc=    0.215  <- OK
(MERLION pid=1915) 05 w=[0.683,0.243,0.074] obj=[   0.20,   0.77,   0.16] sc=    0.333  <- OK
(MERLION pid=1915) 06 w=[0.390,0.302,0.308] obj=[   0.19,   0.77,   0.26] sc=    0.388  <- OK
(MERLION pid=1915) 07 w=[0.049,0.105,0.846] obj=[   0.19,   0.76,   0.14] sc=    0.205  <- OK
(MERLION pid=1915) 08 w=[0.384,0.474,0.142] obj=[   0.20,   0.76,   0.37] sc=    0.489  <- OK
(MERLION pid=1915) 09 w=[0.262,0.519,0.219] obj=[   0.20,   0.75,  

Trial name,adaptation_delta,agent_timesteps_total,connector_metrics,custom_metrics,date,done,episode_len_mean,episode_media,episode_reward_max,episode_reward_max_adapt_1,episode_reward_max_adapt_2,episode_reward_max_adapt_3,episode_reward_max_adapt_4,episode_reward_max_adapt_5,episode_reward_max_adapt_6,episode_reward_max_adapt_7,episode_reward_max_adapt_8,episode_reward_mean,episode_reward_mean_adapt_1,episode_reward_mean_adapt_2,episode_reward_mean_adapt_3,episode_reward_mean_adapt_4,episode_reward_mean_adapt_5,episode_reward_mean_adapt_6,episode_reward_mean_adapt_7,episode_reward_mean_adapt_8,episode_reward_min,episode_reward_min_adapt_1,episode_reward_min_adapt_2,episode_reward_min_adapt_3,episode_reward_min_adapt_4,episode_reward_min_adapt_5,episode_reward_min_adapt_6,episode_reward_min_adapt_7,episode_reward_min_adapt_8,episodes_this_iter,episodes_total,experiment_id,hostname,info,iterations_since_restore,node_ip,num_faulty_episodes,num_healthy_workers,perf,pid,policy_reward_max,policy_reward_mean,policy_reward_min,sampler_perf,time_since_restore,time_this_iter_s,time_total_s,timers,timestamp,timesteps_since_restore,timesteps_this_iter,timesteps_total,training_iteration,trial_id,warmup_time
MERLION_complex_sc_fd7cf_00000,2.84073,0,"{'ObsPreprocessorConnector_ms': 0.004440998632168371, 'StateBufferConnector_ms': 0.003204106364010844, 'ViewRequirementAgentConnector_ms': 0.07759545914147965}",{},2026-02-19_05-43-47,True,100,{},45.1816,41.6111,39.501,45.3355,40.5196,39.0907,42.2278,39.4058,38.4537,31.2326,30.6099,30.3007,33.1519,32.7499,33.5532,36.2871,31.1483,34.0733,15.8438,19.0557,14.3197,21.2469,21.8942,28.1941,24.2573,23.3278,29.7369,777,10101,6d1b934de7fe42c38e02f8c2dee32196,workspace-580f8f1b-0605-48d2-b4f5-8608dc1d24e9-0,"{'learner': {'default_policy': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 1.0, 'worker_loss': 18.21945571899414}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 3600, 'num_grad_updates_lifetime': 754, 'diff_num_grad_updates_vs_sampler_policy': 9.0}}, 'num_steps_sampled': 975600, 'num_steps_trained_this_iter': 81300, 'num_steps_trained': 975600}",13,10.131.58.11,0,5,"{'cpu_util_percent': 8.087387387387388, 'ram_util_percent': 7.895855855855854}",1915,{},{},{},"{'mean_raw_obs_processing_ms': 1.0718007622384746, 'mean_inference_ms': 0.6954476225643293, 'mean_action_processing_ms': 0.1316390780130963, 'mean_env_wait_ms': 2.7645482495950215, 'mean_env_render_ms': 0.0}",5124.96,390.299,5124.96,{},1771479827,1056900,81300,1056900,13,fd7cf_00000,12.8748


(MERLION pid=1915) 
(MERLION pid=1915) === POPULATION OBJECTIVE BREAKDOWN (Iter 1) ===
(MERLION pid=1915) 00 w=[0.028,0.960,0.012] obj=[   0.22,   0.75,   0.15] sc=    0.732  <- OK
(MERLION pid=1915) 01 w=[0.261,0.681,0.058] obj=[   0.19,   0.76,   0.36] sc=    0.588  <- OK
(MERLION pid=1915) 02 w=[0.173,0.583,0.244] obj=[   0.21,   0.76,   0.16] sc=    0.517  <- OK
(MERLION pid=1915) 03 w=[0.436,0.306,0.257] obj=[   0.19,   0.77,   0.33] sc=    0.404  <- OK
(MERLION pid=1915) 04 w=[0.210,0.643,0.146] obj=[   0.20,   0.77,   0.37] sc=    0.591  <- OK
(MERLION pid=1915) 05 w=[0.003,0.131,0.866] obj=[   0.22,   0.77,   0.12] sc=    0.207  <- OK
(MERLION pid=1915) 06 w=[0.723,0.133,0.144] obj=[   0.16,   0.77,   0.26] sc=    0.258  <- OK
(MERLION pid=1915) 07 w=[0.045,0.258,0.696] obj=[   0.20,   0.76,   0.19] sc=    0.335  <- OK
(MERLION pid=1915) 08 w=[0.619,0.237,0.145] obj=[   0.18,   0.77,   0.23] sc=    0.328  <- OK
(MERLION pid=1915) 09 w=[0.395,0.227,0.377] obj=[   0.22,   0.76,  

...


=== Meta-policy 13 ===


(RolloutWorker pid=67626) 2026-02-19 15:26:26,703	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=67626) 2026-02-19 15:26:26,703	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=67626) 2026-02-19 15:26:26,703	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=67626) 2026-02-19 15:26:26,703	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


#### TO TRY NEXT:
* Narrowing the profit range for normalisation >> hopefully when the the agent sees the profit has bigger portion, it affects more to the final rewards, then the agent will consider to put more weights into profit!

#### NOTE, My observation on MERLION behaviour:
It tends to increase the reward on the reward vector value that contributes more in the final scalar reward, which is expected. Consequently, it is sensitive to normalisation range. If we set normalisation range too broad --> initial reard contribution in the beginning will be low --> Merlion thinks that this reward has less significant impact to final scalar reward --> allocate the weights to other reward vector values and decreases the weight for this one.

### METAMORL
timesteps = W (S + 1) H

### MERLION
H = max(timestep/episode, rollout_fragment_length)
S = inner_adaptation_steps
W = num_rollout_workers
P = population_size
J = meta_batch_size

timesteps = W (S + 1) H + P (J (S + 1) H + H)
            {metaMORL-like} {population-stage}

A timestep represents an agent-environment interaction sampling.

In MERLION, the task batch size is defined in meta_batch_size, while num_rollout_workers only determine the parallel computation for speed. In Metamorl, the n_tasks is defined based on num_rollout_worker, so it is attached to the core number used.

In MERLION, the iteration is calculated for the overall population, hence each individual solution learns much less iteration than Metamorl. The justification is because each individual solution learns locally, focusing within narrower search space.

### How the iteration works:

each MERLION individual θᵢ gets fewer total updates than the single MetaMORL meta-policy,

“each individual solution learns locally, focusing within a narrower search space”

because the sample budget is spread across many individuals rather than concentrated on one.

So:

As a theoretical / structural statement: say

“Per global iteration, MERLION trains all individuals; cost scales with P×meta_batch_size.”

As a fair-sample comparison statement: say

“For a fixed environment step budget, each MERLION individual effectively sees fewer updates than a single MetaMORL meta-policy, trading depth per policy for coverage/diversity across the population.”

PS: Total running time in MERLION might more than MetaMORL since aside from metapolicy update (inner and outer), it also involves weight update thru evolutionary mechanism.

NOTE: Currently, MERLION still contains MetaMORL-like update which is actually algorithmically unnecessary but needed. Hence, these stage can be ignored for iteration and timing.

What happens if you “remove Phase 1”? If by “remove Phase 1” you mean: delete inner_adaptation_steps transform, stop producing (samples, metrics_dict), and leave MERLIONMetaUpdate.__call__ untouched,

then:

data_tuple will no longer be a 2-tuple, _shared_samples = data_tuple[0] will be wrong, you’ll get shape/TypeErrors → so yes, bugs.

This is not because of Ray API, but because your execution_plan and MERLIONMetaUpdate are coupled.

#### For now, let's just do interpolation to ignore MetaMORL update and upgrade version later maybe.

### Try to fine-tune using stable-baselines3

In [9]:
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from _merlion.fine_tuning.ft_complex_env_merlion import TestComplexSC
from _metamorl.algo_ray.rllib.algorithms.maml import merlion_finetuning_sb3_independent as ft

In [10]:
# (Re)start Ray cleanly
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

def make_env_ray(config):
    """
    RLlib factory: receives a dict `config` and must return a *Gymnasium Env instance*.
    We wrap with TimeLimit here so both RLlib and Gym share the same horizon behavior.
    """
    base = TestComplexSC(**config)            # pass config keys into your env
    max_steps = int(config.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

register_env("testSC", make_env_ray)
register_env("complex_sc", lambda cfg: complex_env_merlion.make_env(cfg)) # register meta-env

2026-02-21 22:37:00,176	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-02-21 22:37:00,177	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-02-21 22:37:00,179	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

In [12]:
# Optional: re-register safely if re-running cells
try:
    gym.spec("testSC")  # raises if not registered
    # Gymnasium 0.29.x: remove existing spec to allow re-register
    del gym.envs.registration.registry["testSC"]
except Exception:
    pass

def make_env_gym(**kwargs):
    base = TestComplexSC(**kwargs)
    max_steps = int(kwargs.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id="testSC",
    entry_point=make_env_gym,   # callable that returns an Env
)

Finetuning timesteps = P x (M + 1) x K x W

P: population size

M: number of local perturbation

K: env steps per perturbation

W: rollout workers

In [12]:
#finetuning timesteps = 10 x 5 x 100 x 1 = 5000 timesteps
k=6

start_time = time.time()
ft.finetune_with_local_perturbations(
    pop_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/metapolicies-{k}",
    env_id="testSC",
    out_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-{k}",
    total_steps=20000, # default 5000
    record_every=512, # default 1
    eval_episodes=50, # default 50
    m_perturb=2,
    eps=0.05,
    seed=7,
    skip_existing=True,
)
end_time = time.time()
print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

2026-02-19 19:46:19,343	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-02-19 19:46:19,343	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-02-19 19:46:19,359	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2026-02-19 19:46:19,359	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.



=== Meta-policy 0 ===


(RolloutWorker pid=79693) 2026-02-19 19:46:22,572	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=79693) 2026-02-19 19:46:22,572	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=79693) 2026-02-19 19:46:22,572	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=79693) 2026-02-19 19:46:22,572	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
20

[skip+load] i000_m-1: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-6/i000_m-1 already exists, loading existing results into summaries.
[skip+load] i000_m00: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-6/i000_m00 already exists, loading existing results into summaries.
[skip+load] i000_m01: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-6/i000_m01 already exists, loading existing results into summaries.

=== Meta-policy 1 ===


(RolloutWorker pid=80195) 2026-02-19 19:46:35,271	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=80195) 2026-02-19 19:46:35,271	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=80195) 2026-02-19 19:46:35,271	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=80195) 2026-02-19 19:46:35,271	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

...

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.

=== Meta-policy 20 ===


(RolloutWorker pid=89448) 2026-02-19 21:06:18,709	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=89448) 2026-02-19 21:06:18,709	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=89448) 2026-02-19 21:06:18,709	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=89448) 2026-02-19 21:06:18,709	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
Saved:
- /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-6/all_offspring_summary.csv
- /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-6/all_timesteps_long.csv

fine-tuning run-6: 7540.729987859726 seconds


In [ ]:
#finetuning timesteps = 10 x 5 x 100 x 1 = 5000 timesteps
runs = [1, 7]
for k in runs:
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/metapolicies-{k}",
        env_id="testSC",
        out_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-{k}",
        total_steps=20000, # default 5000
        record_every=512, # default 1
        eval_episodes=50, # default 50
        m_perturb=2,
        eps=0.05,
        seed=7,
        skip_existing=True,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

2026-02-17 15:58:59,078	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-02-17 15:58:59,078	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-02-17 15:58:59,117	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2026-02-17 15:58:59,117	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.



=== Meta-policy 0 ===


(RolloutWorker pid=2852) 2026-02-17 15:59:02,234	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=2852) 2026-02-17 15:59:02,234	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=2852) 2026-02-17 15:59:02,234	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=2852) 2026-02-17 15:59:02,234	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
2026-0

[skip+load] i000_m-1: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-1/i000_m-1 already exists, loading existing results into summaries.
[skip+load] i000_m00: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-1/i000_m00 already exists, loading existing results into summaries.
[skip+load] i000_m01: /home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-1/i000_m01 already exists, loading existing results into summaries.

=== Meta-policy 1 ===


(RolloutWorker pid=3354) 2026-02-17 15:59:14,860	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=3354) 2026-02-17 15:59:14,860	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=3354) 2026-02-17 15:59:14,860	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=3354) 2026-02-17 15:59:14,860	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(Rollo

...

/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.

=== Meta-policy 4 ===


(RolloutWorker pid=20894) 2026-02-18 01:15:45,958	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=20894) 2026-02-18 01:15:45,958	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=20894) 2026-02-18 01:15:45,958	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=20894) 2026-02-18 01:15:45,958	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


### Load the model to print policy

In [13]:
MODEL_PATH = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-0/i016_m01/sb3_model.zip"  # <- your file
ENV_ID     = "testSC"
CSV_OUT    = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-0/i016_m01/action_plan.csv"

# 1) Check the env spaces (what you're creating right now)
env = gym.make(ENV_ID, max_steps=100)
print("ENV observation_space:", env.observation_space)
print("ENV action_space     :", env.action_space)

obs, info = env.reset()
print("ENV reset obs shape  :", getattr(obs, "shape", type(obs)))

# 2) Check the model spaces (what it was trained with) WITHOUT attaching an env
model = PPO.load(MODEL_PATH, env=None)
print("MODEL observation_space:", model.observation_space)
print("MODEL action_space     :", model.action_space)

ENV observation_space: Box(0.0, 1.0, (114,), float32)
ENV action_space     : Box(-1.0, 1.0, (64,), float32)
ENV reset obs shape  : (114,)
MODEL observation_space: Box(0.0, 1.0, (114,), float32)
MODEL action_space     : Box(-1.0, 1.0, (64,), float32)


In [25]:
run = 9
MODEL_PATH = f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-{run}/i009_m01/sb3_model.zip"  # <- your file
ENV_ID     = "testSC"
CSV_OUT    = f"/home/rifnyrachman7/_merlion/visualisation/action_com_merlion-{run}.csv"

env = gym.make(ENV_ID, max_steps=100)
model = PPO.load(MODEL_PATH, env=env)

# Try to obtain human-readable action labels, otherwise use a[i]
action_labels = None
ue = getattr(env, "unwrapped", env)
for attr in ("get_action_names", "action_names", "action_labels"):
    if hasattr(ue, attr):
        names = getattr(ue, attr)
        action_labels = names() if callable(names) else names
        break

obs, info = env.reset(seed=7)
rows = []
t = 0
while True:
    action, _ = model.predict(obs, deterministic=True)
    action = np.array(action).ravel()

    row = {"t": t}
    if action_labels and len(action_labels) == action.size:
        for name, val in zip(action_labels, action):
            row[name] = float(val)
    else:
        for i, val in enumerate(action):
            row[f"a{i}"] = float(val)

    # Optional: capture reward vector if the env provides it
    mo = info.get("mo_reward")
    if mo is not None and len(mo) >= 3:
        row["profit"]    = float(mo[0])
        row["emission"]  = float(mo[1])
        row["inequality"]= float(mo[2])

    rows.append(row)

    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    t += 1

env.close()

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print(f"Saved action plan to {CSV_OUT}")

Saved action plan to /home/rifnyrachman7/_merlion/visualisation/action_com_merlion-9.csv
